# Notebook 2 – Modeling, Training, and Evaluation

This notebook trains the CSH-XGB model, evaluates it, and generates plots:
- ROC curve
- Precision–Recall curve
- Confusion matrix
- Threshold vs F2-score
- Feature importance

All figures are saved into `../docs/figures/`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    f1_score,
)

from src.pipeline import train_best_model, load_data, build_preprocessor, hybrid_resampler_steps, compute_class_weights

FIG_DIR = os.path.join("..", "docs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
# Train the best model
trained = train_best_model()
trained.metrics, trained.best_threshold

In [ ]:
# Evaluate on a fresh holdout set for plotting
X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit pipeline again on train (for reproducibility in this notebook)
preprocessor = build_preprocessor(X_train)
class_weights = compute_class_weights(y_train)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    class_weight=class_weights,
    random_state=42,
)

from src.pipeline import _build_xgb_classifier
xgb = _build_xgb_classifier(class_weights)

from src.pipeline import hybrid_resampler_steps
steps = [("pre", preprocessor)]
steps.extend(hybrid_resampler_steps())
steps.append(("clf", xgb))
pipe = ImbPipeline(steps=steps)
pipe.fit(X_train, y_train)

y_scores = pipe.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_scores)
avg_prec = average_precision_score(y_test, y_scores)
roc_auc, avg_prec

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_scores)
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost with Hybrid Sampling')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'roc_curve.png'), dpi=150)
plt.show()

In [ ]:
# Precision-Recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)
ap = average_precision_score(y_test, y_scores)
plt.figure(figsize=(5,4))
plt.step(recalls, precisions, where='post')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (AP = {ap:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pr_curve.png'), dpi=150)
plt.show()

In [ ]:
# Threshold vs F2-score
f2_scores = []
for p, r in zip(precisions, recalls):
    if (p + 4*r) == 0:
        f2_scores.append(0.0)
    else:
        f2_scores.append(5*p*r/(4*p+r))
f2_scores = np.array(f2_scores)
plt.figure(figsize=(5,4))
plt.plot(f2_scores)
plt.title('F2-score across thresholds index')
plt.xlabel('Index in thresholds')
plt.ylabel('F2-score')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'f2_vs_threshold_index.png'), dpi=150)
plt.show()

In [ ]:
# Confusion matrix at trained.best_threshold
best_thr = trained.best_threshold
y_pred_best = (y_scores >= best_thr).astype(int)
cm = confusion_matrix(y_test, y_pred_best)
print(cm)

plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (threshold={best_thr:.3f})')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()